# CLaRa — Fine-Tune on TriviaQA (Kaggle T4)

> **Paper:** *CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning*  
> He et al., Apple / University of Edinburgh — February 2026  
> **Checkpoint:** `tokiggle/clara-7b-e2e-4q` (Kaggle dataset)

---

## Overview

Fine-tunes `query_reasoner_adapter` + `decoder_adapter` on TriviaQA (`rc.nocontext`).

| Component | Status | Rationale |
|-----------|--------|----------|
| `encoder_adapter` | **Frozen** | Compressor pretrained by Apple (Stage I) |
| `query_reasoner_adapter` | **Trained** | Adapts query reasoning |
| `decoder_adapter` | **Trained** | Adapts answer generation |

| Setting | Value |
|---------|-------|
| Train samples | 800 |
| Val samples | 200 |
| LR | 5e-6 (cosine decay) |
| Grad accumulation | 8 |
| Epochs | 1 |
| Dataset variant | rc.nocontext |
| Est. time | ~1.5h |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-A  │  Clone repository & install dependencies
# Run ONCE. After this cell completes → Restart kernel.
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

REPO_URL  = "https://github.com/Duy-Tuyen/introml-clara-implementation.git"
REPO_NAME = "introml-clara-implementation"
REPO_ROOT = f"/kaggle/working/{REPO_NAME}"

print("[1/3] Cloning repository...")
subprocess.run(["rm", "-rf", REPO_ROOT], check=True)
subprocess.run(["git", "clone", "-b", "feature-tuyen2", REPO_URL, REPO_ROOT], check=True)

print("[2/3] Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     f"{REPO_ROOT}/requirements.txt", "-q"],
    check=True,
)

print("[3/3] Running setup script...")
subprocess.run([sys.executable, f"{REPO_ROOT}/setup_env.py"], check=True)

print("\n Setup complete. Please RESTART the kernel before continuing.")

In [ ]:
import shutil, os

cache_path = "/root/.cache/huggingface/modules/transformers_modules"
if os.path.exists(cache_path):
    print("Nuking stale code cache...")
    shutil.rmtree(cache_path)
    print("Cache destroyed.")
else:
    print("Cache was already empty.")

In [ ]:
import os, sys

REPO_ROOT = "/kaggle/working/introml-clara-implementation"
assert os.path.isdir(REPO_ROOT), f"Repository not found at {REPO_ROOT}. Run Cell 0-A first."

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Working directory : {os.getcwd()}")
print(f"Python path entry : {sys.path[0]}")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU: Settings > Accelerator > T4.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU   : {gpu_name}")
print(f"VRAM  : {vram_gb:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1  │  Fine-tune on TriviaQA
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e-4q"
FT_TRIVIAQA_DIR = "/kaggle/working/clara-ft-triviaqa"

print("╔" + "═" * 60 + "╗")
print("║  Fine-Tune CLaRa on TriviaQA (Apple Native Pipeline)       ║")
print("╠" + "═" * 60 + "╣")
print("║  Adapters trained : query_reasoner + decoder (LoRA)         ║")
print("║  Encoder adapter  : FROZEN                                  ║")
print("║  Train/Val        : 800/200  |  LR: 5e-6  |  Epochs: 1     ║")
print("║  Dataset variant  : rc.nocontext (avoids 10GB download)     ║")
print("╚" + "═" * 60 + "╝")

ft_env = os.environ.copy()
ft_env.update({
    "CLARA_CKPT_PATH"    : APPLE_CKPT,
    "CLARA_DATASET"      : "triviaqa",
    "CLARA_N_TRAIN"      : "800",
    "CLARA_N_VAL"        : "200",
    "CLARA_FT_LR"        : "5e-6",
    "CLARA_FT_EPOCHS"    : "1",
    "CLARA_FT_GRAD_ACC"  : "8",
    "CLARA_FT_MAX_DEC_LEN": "128",
    "CLARA_OUTPUT_DIR"   : FT_TRIVIAQA_DIR,
    "CLARA_MODEL_VERSION": "FT_TriviaQA",
})

subprocess.run(
    ["python", "-m", "scripts.finetune_apple"],
    env=ft_env, check=True,
)

print(f"\n✅ Fine-tuning complete. Checkpoint: {FT_TRIVIAQA_DIR}")